# Возбуждение плоского резонатора через impressed current в скин-слое

## Модификация уравнения ротора $H$ для моделирования внешнего тока

---

**Контекст.** Резонатор MenDrive состоит из двух проводящих стенок ($x \le -a$ — левый металл, $x \ge a$ — правый металл) с вакуумным промежутком $-a \le x \le a$. В исходной модели резонатор описывается **однородными** уравнениями Максвелла — задача сводится к поиску собственных частот $\det(M) = 0$.

**Цель этого блокнота.** Ввести в левый металл объёмный сторонний ток (impressed current) $j_{ext}(x)$, распределённый по глубине скин-слоя, и показать, как это меняет:

1. уравнение ротора $\vec H$ (уравнение Ампера),
2. систему граничных условий `sys4`,
3. матрицу `M4_K` и появление вектора правой части $\vec B_{ext}$,
4. структуру решения — переход от задачи на собственные значения к задаче вынужденных колебаний.


## 1. Физическая постановка: что такое impressed current

В электродинамике возбуждения полный ток в правой части уравнения Ампера делится на **две принципиально разные** части:

$$
\nabla \times \vec H = \underbrace{\sigma \vec E}_{j_{cond}\ (\text{закон Ома})} \;+\; \underbrace{\vec j_{ext}}_{\text{impressed current}} \;-\; \frac{i\omega\varepsilon}{c}\vec E
$$

- $j_{cond} = \sigma \vec E$ — ток проводимости, подчиняется закону Ома, зависит от **полного** поля $\vec E$, установившегося в системе;
- $j_{ext}(x)$ — сторонний (импрессионный) ток, **заданный извне**, физически не зависит от $\vec E$ и **не умножается на $\sigma$ среды**.

> **Почему это разграничение критично.** Если попытаться задать возбуждение как «дополнительное поле $E_{ext}$, подставленное в закон Ома $\sigma(E_{ind}+E_{ext})$», получится **двойной счёт**: вклад источника войдёт в правую часть уравнения Ампера дважды — один раз через закон Ома, второй раз как отдельное слагаемое. Корректная постановка требует, чтобы $j_{ext}$ был независимым слагаемым, не связанным с $\sigma$.

Физически $j_{ext}$ моделирует реальный ток возбуждения, накачиваемый в скин-слой металла внешним генератором — независимо от того, какое поле уже установилось в резонаторе.


### Профиль тока в скин-слое

Сторонний ток задаётся затухающим экспоненциальным профилем, аналогичным профилю естественной скин-моды:

$$
j_{ext,y}(x) = j_0 \, e^{\gamma_{ext}(x+a)}, \qquad x \le -a
$$

где $\gamma_{ext}$ обеспечивает затухание при $x \to -\infty$. Для TE-моды (поле направлено вдоль $y$) удобно взять тот же показатель затухания, что и у однородного решения:

$$
\gamma_{ext} = -i K_l
$$

где $K_l$ — волновое число в левом металле, уже найденное из решения $\det(M)=0$. Амплитуда $j_0$ и фаза $\phi_{ext}$ — параметры источника, которые мы контролируем.


## 2. Уравнение ротора $H$ в исходном блокноте (без источника)

В текущей реализации блокнота (TE-мода, диагональный тензор $\mu'$, $\varepsilon'$) уравнение Ампера для левого металла записано в однородном виде:

$$
\left(\nabla \times \vec H\right)_y = -\frac{i\omega}{c}\,\varepsilon'_{l,yy}\, E_y
$$

В коде это уравнение Cell 344:

```python
eq_rot_H_left_conductor_y = rot_H_left_conductor_y == -(I*omega/c * D_l[1])[0]
```

где `D_l = ε'_l · E` — вектор электрического смещения с учётом проводимости через комплексную $\varepsilon'$. Правая часть содержит **только** поляризационный/проводящий отклик среды — никакого стороннего тока здесь нет.


## 3. Модифицированное уравнение ротора $H$ — добавление $j_{ext}$

Вводим объёмный сторонний ток в правую часть уравнения Ампера (в СГС, множитель $4\pi/c$):

$$
\boxed{\left(\nabla \times \vec H\right)_y = -\frac{i\omega}{c}\,\varepsilon'_{l,yy}\, E_y \;+\; \frac{4\pi}{c}\, j_{ext,y}(x)}
$$

Для TE-моды это единственная компонента ротора $H$, которая входит в систему уравнений (остальные компоненты $j_{ext,x} = j_{ext,z} = 0$, так как ток течёт вдоль $y$, как и собственное поле $E_y$).

Важно: **уравнения ротора $H$ для вакуума и для правого металла не меняются** — источник присутствует только в объёме левого металла.



```python
# ============================================================================
# СИМВОЛЬНОЕ ПРЕДСТАВЛЕНИЕ j_ext
# ============================================================================
# Эта ячейка вставляется ПЕРЕД определением eq_rot_H_left_conductor
# (то есть до Cell ~344 в исходном блокноте)

j0_ext = var('j0_ext', domain='complex')   # амплитуда импрессионного тока [СГС: с⁻¹]
j_ext_y_sym = function('j_ext_y')(x)        # символьная функция профиля тока

# Конкретный профиль будет подставлен позже:
#   j_ext_y_sym(x) → j0_ext * exp(-I * K_left_conductor * (x + a))
# то есть тот же скин-профиль, что у собственной TE-моды резонатора

```


```python
# ============================================================================
# МОДИФИЦИРОВАННОЕ УРАВНЕНИЕ РОТОРА H ДЛЯ ЛЕВОГО МЕТАЛЛА (TE)
# Заменяет исходную ячейку eq_rot_H_left_conductor_y (Cell 344)
# ============================================================================

eq_rot_H_left_conductor_x_ext = rot_H_left_conductor_x == -(I*omega/c*D_l[0])[0]

eq_rot_H_left_conductor_y_ext = rot_H_left_conductor_y == \
    -(I*omega/c*D_l[1])[0] + 4*pi/c * j_ext_y_sym

eq_rot_H_left_conductor_z_ext = rot_H_left_conductor_z == -(I*omega/c*D_l[2])[0]

eq_rot_H_left_conductor_ext = [
    eq_rot_H_left_conductor_x_ext,
    eq_rot_H_left_conductor_y_ext,
    eq_rot_H_left_conductor_z_ext,
]

print("Модифицированное уравнение ротора H (компонента y):")
show(eq_rot_H_left_conductor_y_ext)

# Уравнения для вакуума (eq_rot_H_inner_vacuum, Cell 343)
# и для правого металла (eq_rot_H_right_conductor, Cell 346)
# остаются БЕЗ ИЗМЕНЕНИЙ — источник присутствует только в левом металле

```

## 4. Вывод неоднородного волнового уравнения

Применим тот же путь исключения переменных, что и в исходном блокноте, но теперь с модифицированным уравнением ротора $H$.

### Шаг 1. Берём ротор от уравнения Фарадея

$$
\nabla \times (\nabla \times \vec E) = \frac{i\omega\mu'_{l}}{c}(\nabla \times \vec H)
$$

Подставляем модифицированное уравнение ротора $H$:

$$
\nabla \times (\nabla \times \vec E) = \frac{i\omega\mu'_{l,zz}}{c}\left[-\frac{i\omega}{c}\varepsilon'_{l,yy} E_y + \frac{4\pi}{c} j_{ext,y}\right]
$$

### Шаг 2. Раскрываем и используем $\nabla\cdot\vec E \approx 0$

$$
-\nabla^2 E_y = \frac{\omega^2}{c^2}\mu'_{l,zz}\varepsilon'_{l,yy}\, E_y + \frac{4\pi i \omega \mu'_{l,zz}}{c^2}\, j_{ext,y}
$$

### Шаг 3. Переносим, вводим $K_l^2 = \dfrac{\omega^2}{c^2}\mu'_{l,zz}\varepsilon'_{l,yy}$

$$
\boxed{\frac{d^2 E_y}{dx^2} + K_l^2\, E_y = -\frac{4\pi i \omega \mu'_{l,zz}}{c^2}\, j_{ext,y}(x)}
$$

Это **неоднородное уравнение Гельмгольца**. Правая часть отлична от нуля только в левом металле, где течёт $j_{ext}$.


## 5. Частное решение для экспоненциального профиля тока

Если $j_{ext,y}(x) = j_0\, e^{\gamma_{ext}(x+a)}$, ищем частное решение в той же форме:

$$
E_{y,part}(x) = P \, e^{\gamma_{ext}(x+a)}
$$

Подставляя в волновое уравнение:

$$
\gamma_{ext}^2 P + K_l^2 P = -\frac{4\pi i \omega \mu'_{l,zz}}{c^2}\, j_0
$$

$$
\boxed{P = \frac{-4\pi i \omega \mu'_{l,zz}\, j_0 / c^2}{\gamma_{ext}^2 + K_l^2}}
$$

Знаменатель $\gamma_{ext}^2 + K_l^2 \ne 0$ при стандартном скин-профиле ($\gamma_{ext} = -iK_l$ даёт $\gamma_{ext}^2 = -K_l^2$ только в пределе нулевых потерь — для комплексного $K_l$ с конечной мнимой частью это вырождение не возникает в общем случае).

Магнитное поле от частного решения находится из уравнения Фарадея:

$$
H_{z,part}(x) = \frac{c}{i\omega\mu'_{l,zz}}\,\frac{dE_{y,part}}{dx} = \frac{c\,\gamma_{ext}}{i\omega\mu'_{l,zz}}\, P\, e^{\gamma_{ext}(x+a)}
$$


## 6. Как меняется система граничных условий `sys4`

В исходном блокноте (TE-мода, Cell 476):

```python
sys4 = [eqHz_l, eqHz_r, eq_Ey_l, eq_Ey_r]
```

где:

- `eqHz_l`: $H_z^{left}(-a) = H_z^{vacuum}(-a)$ — непрерывность $H_z$ слева;
- `eq_Ey_l`: $E_y^{left}(-a) = E_y^{vacuum}(-a)$ — непрерывность $E_y$ слева;
- аналогично `eqHz_r`, `eq_Ey_r` справа.

### Что меняется

**Условие непрерывности $H_z$ не меняется.** Поскольку $j_{ext}$ — объёмный (не поверхностный) ток, скачка тангенциального $H_z$ на самой границе $x=-a$ не возникает — ток распределён по глубине, а не сконцентрирован в бесконечно тонком слое.

**Условие на $E_y$ меняется**, потому что выражение $E_y^{left}(x)$ теперь содержит добавку от частного решения:

$$
E_y^{left}(x) = \underbrace{C_l\, e^{iK_l(x+a)}}_{\text{однородное (как раньше)}} + \underbrace{P\, e^{\gamma_{ext}(x+a)}}_{\text{частное, от }j_{ext}}
$$

При подстановке в граничное условие при $x=-a$ ($e^{\gamma_{ext}\cdot 0}=1$):

$$
C_l + P = E_y^{vacuum}(-a)
$$

Слагаемое $P$ — известная величина (выражается через $j_0$), поэтому при переносе в правую часть оно становится источником неоднородности всей системы.



```python
# ============================================================================
# ПЕРЕСЧЁТ E_y_l С УЧЁТОМ j_ext (символьно, через существующую get_field())
# Использует функцию get_field(), уже определённую в блокноте (Cell 349)
# ============================================================================

E_left_conductor_sb_y_ext = get_field(
    eq_rot_E_left_conductor,
    eq_rot_H_left_conductor_ext,      # модифицированное уравнение ротора H
    eq_rot_E_index,
    eq_rot_H_index,
    field_for_exclude,
    field_for_solve
)

print("E_y в левом металле с учётом j_ext:")
show(E_left_conductor_sb_y_ext)

```


```python
# ============================================================================
# ГРАНИЧНОЕ УСЛОВИЕ eq_Ey_l С j_ext
# Заменяет исходную ячейку eq_Ey_l (Cell 416)
# ============================================================================

eq_Ey_l_ext = (E_left_conductor_sb_y_ext.rhs().subs(x == -a)
               == E_inner_vacuum_sub_y(-a))

print("Модифицированное граничное условие на E_y (левая стенка):")
show(eq_Ey_l_ext)

# Граничные условия eqHz_l, eqHz_r, eq_Ey_r — БЕЗ ИЗМЕНЕНИЙ
# (j_ext объёмный, на правой стенке отсутствует)

```

## 7. Модифицированная система `sys4_ext` и разделение на матрицу и источник

Полная система с источником:

```python
sys4_ext = [eqHz_l, eqHz_r, eq_Ey_l_ext, eq_Ey_r]
```

Принципиальный момент: `GenerateMatrix()` (как и в исходном блокноте) строит матрицу коэффициентов **только при неизвестных** $(B_{1zl}, A_{1z}, A_{2z}, B_{1zr})$. Слагаемое с $j_0$ не содержит этих неизвестных — оно становится **вектором правой части** $\vec B_{ext}$:

$$
M_4 \cdot \vec X = \vec B_{ext}(j_0)
$$

где $M_4$ — **та же самая матрица**, что в однородной задаче (без источника). Это ключевое свойство: резонансный знаменатель $\det(M_4) = 0$ не меняется — частоты собственных колебаний резонатора остаются теми же. Меняется только то, что теперь решение существует при любой $\omega$, и **резко возрастает** при приближении к резонансной частоте.



```python
# ============================================================================
# МОДИФИЦИРОВАННАЯ sys4_ext, МАТРИЦА M4_K_ext И ВЕКТОР B_ext
# ============================================================================

if is_TE:
    sys4_ext = [
        eqHz_l,        # непрерывность Hz — без изменений
        eqHz_r,        # непрерывность Hz — без изменений
        eq_Ey_l_ext,   # E_y на левой границе, теперь с j_ext
        eq_Ey_r,       # правая граница — без изменений
    ]

# Однородная матрица (коэффициенты при неизвестных, j_ext = 0)
M4_K_ext = GenerateMatrix(
    [eq.subs(j0_ext == 0) for eq in sys4_ext],
    vars4
)

# Вектор правой части — то, что остаётся при нулевых неизвестных
B_ext_symbolic = vector(SR, [
    -(eq.lhs() - eq.rhs()).subs(j0_ext == 0).subs({v: 0 for v in vars4})
    for eq in sys4_ext
])

print("Матрица M4_K_ext (совпадает с исходной M4_K при j0_ext=0):")
show(M4_K_ext)

print("\nВектор правой части B_ext (источник от j0_ext):")
show(B_ext_symbolic)

```

## 8. Проверка: матрица не изменилась

Контрольная проверка корректности патча — матрица `M4_K_ext` должна совпадать с исходной `M4_K` из однородной задачи:



```python
# Проверка совпадения матриц (должно быть True)
matrices_match = (M4_K_ext - M4_K).expand().is_zero()
print(f"M4_K_ext == M4_K (детерминант, резонансные частоты не меняются): {matrices_match}")

if not matrices_match:
    print("ВНИМАНИЕ: матрицы не совпадают — проверьте корректность патча!")
    show((M4_K_ext - M4_K).expand())

```

## 9. Численное решение неоднородной системы

После подстановки числовых значений параметров (`digit_values`, `k_z_sol`, `epsilon_mu_subs`) и конкретного профиля $j_{ext}$:

$$
\vec X = M_4^{-1}\, \vec B_{ext}(j_0)
$$

В коде это заменяет поиск $\det(M_4)=0$ на прямое решение `M4.solve_right(B_ext)`. При сканировании $\omega$ вблизи резонансной частоты $\omega_{res}$ (где $\det(M_4) \to 0$) амплитуды коэффициентов $B_{1zl}, A_{1z}, A_{2z}, B_{1zr}$ резко возрастают, ограничиваясь только конечной добротностью резонатора (мнимой частью $\omega_{res}$, определяемой потерями в стенках).



```python
# ============================================================================
# ЧИСЛЕННОЕ РЕШЕНИЕ: подстановка digit_values и решение M4 * X = B_ext
# ============================================================================

# Подставляем числовые значения в M4_K_ext и B_ext_symbolic
M4_ext_d = M4_K_ext \
    .subs(K_vacuum_subs) \
    .subs(K_left_conductor_subs) \
    .subs(K_right_conductor_subs) \
    .subs(kappa_vacuum_sol) \
    .subs(epsilon_mu_subs) \
    .subs(digit_values_init) \
    .subs(k_z_sol)

# Профиль j_ext: подставляем конкретный скин-профиль вместо символьной функции
j0_value = complex128(1.0)   # амплитуда источника (варьируется)

B_ext_d = B_ext_symbolic \
    .subs(K_vacuum_subs) \
    .subs(K_left_conductor_subs) \
    .subs(K_right_conductor_subs) \
    .subs(kappa_vacuum_sol) \
    .subs(epsilon_mu_subs) \
    .subs(digit_values_init) \
    .subs(k_z_sol) \
    .subs(j0_ext == j0_value)

print("Численная матрица M4_ext_d:")
show(M4_ext_d)

print("\nЧисленный вектор B_ext_d:")
show(B_ext_d)

# Решение неоднородной СЛАУ
X_sol_ext = M4_ext_d.solve_right(B_ext_d)
B1_zl_sol, A1_z_sol, A2_z_sol, B1_zr_sol = X_sol_ext

print(f"\nB1_zl (амплитуда Hz в левом металле) = {B1_zl_sol}")
print(f"A1_z, A2_z (амплитуды в вакууме)       = {A1_z_sol}, {A2_z_sol}")
print(f"B1_zr (амплитуда Hz в правом металле)  = {B1_zr_sol}")

```

## 10. Влияние на расчёт пондеромоторной силы

Полное поле в левом металле теперь содержит вклад от $j_{ext}$:

$$
E_y^{left}(x) = C_l\, e^{iK_l(x+a)} + P\, e^{\gamma_{ext}(x+a)}, \qquad
H_z^{left}(x) = H_{z,homo}(x) + H_{z,part}(x)
$$

Полный ток проводимости: $j_{cond,y} = \sigma_{e,l,yy}\, E_y^{left}$. Сила Лоренца:

$$
\langle f_x\rangle = \frac{1}{2c}\,\mathrm{Re}\!\left[\left(\sigma_{e,l,yy}E_y^{left} + j_{ext,y}\right)\cdot \overline{\mu'_{l,zz}H_z^{left}}\right]
$$

Раскрывая, получаем **четыре физически различных члена**:

| Член | Физический смысл |
|---|---|
| $\sigma E_{homo}\cdot \overline{\mu H_{homo}}$ | токи Фуко в резонаторе (исходный расчёт `calc_p_Ht_left_jeH_conj`) |
| $j_{ext}\cdot \overline{\mu H_{homo}}$ | **основная резонансная тяга** — растёт на резонансе |
| $\sigma E_{homo}\cdot \overline{\mu H_{part}}$ | перекрёстный член |
| $j_{ext}\cdot \overline{\mu H_{part}}$ | самодействие источника |

Численная реализация этого расчёта (с оптимизацией фазы $\phi_{ext}$ для максимизации второго члена) вынесена в отдельный файл `j_ext_impressed_current.py` — он использует частное решение $P$, выведенное здесь, напрямую, без необходимости пересобирать символьную матрицу.


## 11. Резюме изменений

| Элемент | Было (однородная задача) | Стало (с $j_{ext}$) |
|---|---|---|
| Уравнение ротора $H$ (левый металл, $y$) | $-\dfrac{i\omega}{c}\varepsilon'_{l,yy}E_y$ | $+\dfrac{4\pi}{c}j_{ext,y}(x)$ |
| Волновое уравнение для $E_y$ | однородное Гельмгольца | неоднородное, источник $\propto j_{ext}$ |
| Граничное условие $H_z$ (левая стенка) | непрерывность | **без изменений** (ток объёмный) |
| Граничное условие $E_y$ (левая стенка) | непрерывность | непрерывность $+$ вклад частного решения $P$ |
| Матрица $M_4$ | определяет $\det(M_4)=0$ | **та же самая** |
| Тип задачи | на собственные значения | вынужденных колебаний, $M_4\vec X = \vec B_{ext}$ |
| Резонансные частоты | корни $\det(M_4)=0$ | **не изменились** |
| Сила Лоренца | один член ($\sigma E\cdot \overline{\mu H}$) | четыре члена, включая резонансную тягу $j_{ext}\cdot\overline{\mu H}$ |

